# Lenguaje de Programación Visual - FIUNA
## Semana 5: Construcción de APIs Propias con FastAPI y Pydantic

**Profesor:** Jorge Luis Tillería Mereles  
**Auxiliar de Práctica:** Carlos María Benítez Cardozo  
**Carrera:** Ingeniería Mecatrónica  
**Ciclo:** 2026-02  

---

### Objetivos de la Sesión:
1. Comprender la arquitectura de **microservicios REST modernos** basados en **FastAPI** y servidores ASGI.
2. Implementar esquemas estrictos de validación, tipado y coerción de datos utilizando **Pydantic v2**.
3. Aplicar validadores mecatrónicos personalizados con `@field_validator` y restricciones con `Field`.
4. Diseñar e interactuar con endpoints CRUD para estaciones de monitoreo físico y telemetría.
5. Explorar la documentación interactiva OpenAPI autogenerada (**Swagger UI** y **ReDoc**).

## 1. Validación de Datos con Pydantic v2

**Pydantic** es una librería de parseo y validación de tipos basada en anotaciones de Python (`type hints`). A diferencia de los dataclasses tradicionales, Pydantic garantiza que los datos entrantes cumplan estrictamente con las restricciones físicas y de formato antes de ingresar a la lógica del sistema.

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field, ValidationError, field_validator

class TipoSensor(str, Enum):
    TEMPERATURA = "temperatura"
    PRESION = "presion"
    VIBRACION = "vibracion"

class SensorSchema(BaseModel):
    nombre: str = Field(..., min_length=3, max_length=50)
    tipo: TipoSensor
    ubicacion: str = Field(..., min_length=2)
    unidad: str
    umbral_alerta: float = Field(..., gt=0, description="El umbral debe ser un valor positivo")

    @field_validator("nombre")
    @classmethod
    def limpiar_nombre(cls, valor: str) -> str:
        limpio = valor.strip()
        if not limpio:
            raise ValueError("El nombre no puede estar compuesto únicamente de espacios")
        return limpio

# Prueba de validación exitosa
sensor_valido = SensorSchema(
    nombre="Sensor Térmico Reactor 1",
    tipo=TipoSensor.TEMPERATURA,
    ubicacion="Celda de Manufactura 4",
    unidad="°C",
    umbral_alerta=95.5
)
print("Sensor validado exitosamente:")
print(sensor_valido.model_dump_json(indent=2))

# Prueba de captura de error de validación
try:
    SensorSchema(
        nombre="   ",  # Nombre inválido
        tipo="velocidad", # Tipo no existente en Enum
        ubicacion="A",
        unidad="rpm",
        umbral_alerta=-10.0 # Umbral negativo inválido
    )
except ValidationError as e:
    print("\nErrores detectados por Pydantic:")
    for err in e.errors():
        print(f"- Campo '{err['loc'][0]}': {err['msg']}")

## 2. Pruebas Interactivas de la API con `TestClient`

FastAPI integra `TestClient` (construido sobre `httpx`), lo que permite realizar peticiones HTTP directamente a la aplicación en memoria sin necesidad de levantar un servidor externo en un proceso separado.

In [ ]:
import sys
from pathlib import Path

# Ajustar ruta para importar el paquete src
sys.path.insert(0, str(Path.cwd()))

from fastapi.testclient import TestClient
from src.app import app

client = TestClient(app)

# 1. Verificar estado de la API
res = client.get("/")
print(f"Estado HTTP: {res.status_code}")
print("Respuesta:", res.json())

### 2.1 Registro de Sensores Mecatrónicos (`POST /sensores`)

In [ ]:
nuevo_sensor = {
    "nombre": "Acelerómetro Brazo Robótico",
    "tipo": "vibracion",
    "ubicacion": "Eje J3 - Robot Industrial",
    "unidad": "m/s²",
    "umbral_alerta": 4.5
}

res_creacion = client.post("/sensores", json=nuevo_sensor)
print(f"Código de Respuesta: {res_creacion.status_code}")
sensor_creado = res_creacion.json()
sensor_id = sensor_creado["id"]
print("Sensor registrado en base de datos:", sensor_creado)

### 2.2 Simulación de Adquisición de Telemetría (`POST /sensores/{id}/lecturas`)

Registramos una serie de lecturas físicas y observamos cómo el microservicio evalúa automáticamente el umbral de seguridad para disparar alertas.

In [ ]:
lecturas_muestra = [
    {"valor": 1.2, "observacion": "Operación nominal"},
    {"valor": 2.8, "observacion": "Aceleración en trayectoria"},
    {"valor": 5.1, "observacion": "Pico por impacto - ALERTA ESPERADA"},
    {"valor": 2.0, "observacion": "Retorno a régimen estable"}
]

for lectura in lecturas_muestra:
    r = client.post(f"/sensores/{sensor_id}/lecturas", json=lectura)
    datos = r.json()
    alerta_str = "🚨 ALERTA ACTIVA" if datos["alerta_activa"] else "✅ Normal"
    print(f"Lectura: {datos['valor']} {datos['unidad']} | Estado: {alerta_str} | Nota: {datos['observacion']}")

### 2.3 Consulta del Resumen Estadístico (`GET /sensores/{id}/resumen`)

In [ ]:
res_resumen = client.get(f"/sensores/{sensor_id}/resumen")
resumen = res_resumen.json()

print("=" * 60)
print(f"REPORTE DE LA ESTACIÓN: {resumen['sensor']['nombre']}")
print(f"Total de Lecturas : {resumen['total_lecturas']}")
print(f"Valor Promedio    : {resumen['promedio_valor']} {resumen['sensor']['unidad']}")
print(f"Valor Máximo      : {resumen['maximo_valor']} {resumen['sensor']['unidad']}")
print(f"Alertas Emitidas  : {resumen['alertas_registradas']}")
print("=" * 60)

## 3. Inspección del Contrato OpenAPI Autogenerado

Una de las fortalezas centrales de **FastAPI** es que genera automáticamente el estándar **OpenAPI 3.1.0** a partir de los modelos Pydantic y las firmas de funciones de ruta. Esto alimenta interfaces interactivas como **Swagger UI** (`/docs`) y **ReDoc** (`/redoc`).

In [ ]:
schema_openapi = app.openapi()
print(f"Título de la API : {schema_openapi['info']['title']}")
print(f"Versión de OpenAPI: {schema_openapi['openapi']}")
print(f"Total de Rutas   : {len(schema_openapi['paths'])}")
print("Endpoints registrados:")
for path, metodos in schema_openapi['paths'].items():
    for metodo in metodos:
        print(f"- {metodo.upper()} {path}")